# Load and Combine Each Patient File

In [4]:
import os
import pandas as pd
from glob import glob
from tqdm import tqdm

# 1. Load all PSV files

data_path = "C:\\Users\\imama\\Documents\\GitHub\\safe_dss\\dataset_raw"
psv_files = glob(os.path.join(data_path, "*.psv"))

print(f"Total files found: {len(psv_files)}")

all_patients = []

for file in tqdm(psv_files):
    try:
        df = pd.read_csv(file, sep='|')
        
        # Extract patient ID from filename
        patient_id = os.path.basename(file).replace(".psv", "")
        df["patient_id"] = patient_id
        
        all_patients.append(df)
        
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Combine all patients into one dataframe
full_df = pd.concat(all_patients, ignore_index=True)

print("Combined dataset shape:", full_df.shape)

Total files found: 20336


100%|██████████| 20336/20336 [04:28<00:00, 75.67it/s] 


Combined dataset shape: (790215, 42)


In [ ]:
# full_df.to_csv('full_patient_data.csv', index=False)
full_df.head(5)

,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,patient_id
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,1,0,p000001
1,97.0,95.0,NaN,98.0,75.33,NaN,19.0,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,2,0,p000001
2,89.0,99.0,NaN,122.0,86.00,NaN,22.0,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,3,0,p000001
3,90.0,95.0,NaN,NaN,NaN,NaN,30.0,NaN,24.0,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,4,0,p000001
4,103.0,88.5,NaN,122.0,91.33,NaN,24.5,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,5,0,p000001


# Stratified Balanced Sampling

In [7]:
patient_labels = (
    full_df.groupby("patient_id")["SepsisLabel"]
    .max()
    .reset_index()
)

patient_labels.columns = ["patient_id", "patient_sepsis_flag"]

print(patient_labels["patient_sepsis_flag"].value_counts())

patient_sepsis_flag
0    18546
1     1790
Name: count, dtype: int64


In [8]:
# Separate positive and negative patients
positive_patients = patient_labels[patient_labels["patient_sepsis_flag"] == 1]
negative_patients = patient_labels[patient_labels["patient_sepsis_flag"] == 0]

print("Positive patients:", len(positive_patients))
print("Negative patients:", len(negative_patients))

# Choose equal numbers
n_samples = 1790  # balanced

positive_sample = positive_patients.sample(n=n_samples, random_state=42)
negative_sample = negative_patients.sample(n=n_samples, random_state=42)

balanced_patients = pd.concat([positive_sample, negative_sample])

print("Final patient count:", len(balanced_patients))

Positive patients: 1790
Negative patients: 18546
Final patient count: 3580


In [ ]:
balanced_patient_ids = balanced_patients["patient_id"].tolist()

balanced_df = full_df[full_df["patient_id"].isin(balanced_patient_ids)]

print("Balanced dataset shape:", balanced_df.shape)
# balanced_df.to_csv('balanced_patient_data.csv', index=False)

Balanced dataset shape: (171736, 42)
